In [ ]:

# 1. 导入必要的库
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


In [ ]:

# 2. 加载数据
train_csv_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/wild_blueberry_yield/train.csv'
train_data = pd.read_csv(train_csv_path)

# 3. 数据预处理
print(train_data.head())
print(train_data.describe())
print(train_data.info())


      id  clonesize  honeybee  ...  fruitmass      seeds       yield
0    245       25.0      0.50  ...   0.399724  29.742583  4177.01520
1   3017       12.5      0.25  ...   0.376874  27.735098  3238.02815
2   8047       12.5      0.25  ...   0.488569  40.655498  7451.72563
3  14223       12.5      0.25  ...   0.466671  37.966864  6580.39696
4  13397       25.0      0.50  ...   0.453650  36.600113  6019.26248

[5 rows x 18 columns]
                 id     clonesize  ...         seeds         yield
count  12231.000000  12231.000000  ...  12231.000000  12231.000000
mean    7655.390074     19.694628  ...     36.193681   6030.165882
std     4421.960747      6.562779  ...      4.045115   1339.792879
min        1.000000     10.000000  ...     22.079199   1945.530610
25%     3817.000000     12.500000  ...     33.232449   5126.993180
50%     7641.000000     25.000000  ...     36.047770   6117.475900
75%    11507.500000     25.000000  ...     39.203069   7028.673500
max    15286.000000     40.

In [ ]:


# 4. 拆分数据集（训练集和测试集）
features = train_data.drop(columns=['id', 'yield'])  # 'id' is not needed, 'yield' is the target variable
target = train_data['yield']

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# 5. 选择模型
model = RandomForestRegressor(random_state=42)

# 6. 训练模型
model.fit(X_train, y_train)

# 7. 评估模型
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)

print(f"Mean Absolute Error (MAE) on the test set: {mae:.2f}")



Mean Absolute Error (MAE) on the test set: 390.30


In [ ]:


# 8. 超参数调优（使用Grid Search）
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, 
                           scoring='neg_mean_absolute_error', cv=5, n_jobs=-1, verbose=2)

grid_search.fit(X_train, y_train)

# 打印最佳参数和最佳得分
print("Best Parameters:", grid_search.best_params_)
print("Best MAE:", -grid_search.best_score_)

# 使用最佳参数重新训练模型
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)

print(f"Mean Absolute Error (MAE) on the test set with best parameters: {mae:.2f}")



Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Parameters: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 200}
Best MAE: 349.1954494868693
Mean Absolute Error (MAE) on the test set with best parameters: 370.37


In [ ]:



# 9. 特征工程（检查多重共线性）
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data['feature'] = features.columns
vif_data['VIF'] = [variance_inflation_factor(features.values, i) for i in range(len(features.columns))]

print(vif_data)

# 查看VIF值大于10的特征（通常认为VIF>10表示高度共线性）
high_vif_features = vif_data[vif_data['VIF'] > 10]['feature'].tolist()
print("Features with VIF > 10:", high_vif_features)




---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
Cell In[6], line 4
      2 pass
      3 print('##active_line2##')
----> 4 from statsmodels.stats.outliers_influence import variance_inflation_factor
      5 print('##active_line3##')
      6 pass

ModuleNotFoundError: No module named 'statsmodels'

In [ ]:



# 安装statsmodels库
!pip install statsmodels



D:\software\Python\PyCharm\PyCharm 20250201\PyCharm 2025.2.0.1\plugins\python-ce\helpers\pycharm_display\datalore\display\supported_data_type.py:6: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  import numpy


In [ ]:


import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 带有VIF值的特征DataFrame
vif_data = pd.DataFrame()
vif_data['feature'] = features.columns
vif_data['VIF'] = [variance_inflation_factor(features.values, i) for i in range(len(features.columns))]

print(vif_data)

# 查看VIF值大于10的特征（通常认为VIF>10表示高度共线性）
high_vif_features = vif_data[vif_data['VIF'] > 10]['feature'].tolist()
print("Features with VIF > 10:", high_vif_features)




---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
Cell In[8], line 4
      2 import numpy as np
      3 print('##active_line2##')
----> 4 from statsmodels.stats.outliers_influence import variance_inflation_factor
      5 print('##active_line3##')
      6 pass

ModuleNotFoundError: No module named 'statsmodels'

In [ ]:



# 确认当前Python环境
import sys
print("Current Python environment:", sys.executable)

# 尝试重新安装statsmodels以确保其可用
!{sys.executable} -m pip install statsmodels --upgrade --force-reinstall

# 重新导入statsmodels
import statsmodels.api as sm

# 重新检查VIF
vif_data = pd.DataFrame()
vif_data['feature'] = features.columns
vif_data['VIF'] = [variance_inflation_factor(features.values, i) for i in range(len(features.columns))]

print(vif_data)

# 查看VIF值大于10的特征（通常认为VIF>10表示高度共线性）
high_vif_features = vif_data[vif_data['VIF'] > 10]['feature'].tolist()
print("Features with VIF > 10:", high_vif_features)


Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

error.html

Please note and check the following:

  * The Python version is: Python3.10 from "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\Scripts\python.exe"
  * The NumPy version is: "1.24.3"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: DLL load failed while importing _multiarray_umath: 找不到指定的模块。
---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
Cell In[9], line 7
      5 get_ipython().system('{sys.executable} -m pip install statsmodels --upgrade --force-reinstall')
      6 # 重新导入statsmodels
----> 7 import statsmodels.api as sm
      8 # 重新检查VIF
      9 vif_data = pd.DataFrame()

File D:\LLM-Driven_AI

In [ ]:



import sys

# 卸载NumPy
!{sys.executable} -m pip uninstall numpy -y

# 重新安装NumPy
!{sys.executable} -m pip install numpy==1.21.5

# 重新安装statsmodels
!{sys.executable} -m pip install statsmodels --upgrade --force-reinstall

# 重新导入statsmodels
import statsmodels.api as sm

# 重新检查VIF
vif_data = pd.DataFrame()
vif_data['feature'] = features.columns
vif_data['VIF'] = [variance_inflation_factor(features.values, i) for i in range(len(features.columns))]

print(vif_data)

# 查看VIF值大于10的特征（通常认为VIF>10表示高度共线性）
high_vif_features = vif_data[vif_data['VIF'] > 10]['feature'].tolist()
print("Features with VIF > 10:", high_vif_features)



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

rror.html

Please note and check the following:

  * The Python version is: Python3.10 from "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\Scripts\python.exe"
  * The NumPy version is: "1.24.3"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: DLL load failed while importing _multiarray_umath: 找不到指定的模块。
---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
Cell In[10], line 9
      7 get_ipython().system('{sys.executable} -m pip install statsmodels --upgrade --force-reinstall')
      8 # 重新导入statsmodels
----> 9 import statsmodels.api as sm
     10 # 重新检查VIF
     11 vif_data = pd.DataFrame()

File D:\LLM-Driven_AI

In [ ]:



import sys

# 卸载NumPy
!{sys.executable} -m pip uninstall numpy -y

# 重新安装NumPy
!{sys.executable} -m pip install numpy==1.21.5

# 重新安装statsmodels
!{sys.executable} -m pip install statsmodels --upgrade --force-reinstall

# 重新导入statsmodels
import statsmodels.api as sm

# 重新检查VIF
vif_data = pd.DataFrame()
vif_data['feature'] = features.columns
vif_data['VIF'] = [variance_inflation_factor(features.values, i) for i in range(len(features.columns))]

print(vif_data)

# 查看VIF值大于10的特征（通常认为VIF>10表示高度共线性）
high_vif_features = vif_data[vif_data['VIF'] > 10]['feature'].tolist()
print("Features with VIF > 10:", high_vif_features)




Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

rror.html

Please note and check the following:

  * The Python version is: Python3.10 from "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\Scripts\python.exe"
  * The NumPy version is: "1.24.3"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: DLL load failed while importing _multiarray_umath: 找不到指定的模块。
---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
Cell In[11], line 9
      7 get_ipython().system('{sys.executable} -m pip install statsmodels --upgrade --force-reinstall')
      8 # 重新导入statsmodels
----> 9 import statsmodels.api as sm
     10 # 重新检查VIF
     11 vif_data = pd.DataFrame()

File D:\LLM-Driven_AI

In [ ]:




import sys

# 卸载NumPy
!{sys.executable} -m pip uninstall numpy -y

# 重新安装NumPy（版本1.21.5）
!{sys.executable} -m pip install numpy==1.21.5 --force-reinstall

# 等待NumPy安装完成
import time
time.sleep(10)

# 重新安装statsmodels
!{sys.executable} -m pip install statsmodels --upgrade --force-reinstall

# 等待statsmodels安装完成
time.sleep(10)

# 重新导入statsmodels
import statsmodels.api as sm

# 重新检查VIF
vif_data = pd.DataFrame()
vif_data['feature'] = features.columns
vif_data['VIF'] = [variance_inflation_factor(features.values, i) for i in range(len(features.columns))]

print(vif_data)

# 查看VIF值大于10的特征（通常认为VIF>10表示高度共线性）
high_vif_features = vif_data[vif_data['VIF'] > 10]['feature'].tolist()
print("Features with VIF > 10:", high_vif_features)




Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

bleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python3.10 from "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\Scripts\python.exe"
  * The NumPy version is: "1.24.3"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: DLL load failed while importing _multiarray_umath: 找不到指定的模块。
---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
Cell In[12], line 14
     12 time.sleep(10)
     13 # 重新导入statsmodels
---> 14 import statsmodels.api as sm
     15 # 重新检查VIF
     16 vif_data = pd.DataFrame()

File D:\LLM-Driven_AI

In [ ]:





import sys

# 卸载NumPy
!{sys.executable} -m pip uninstall numpy -y

# 重新安装NumPy（版本1.21.5）
!{sys.executable} -m pip install numpy==1.21.5 --force-reinstall

# 等待NumPy安装完成
import time
time.sleep(10)

# 重新安装statsmodels
!{sys.executable} -m pip install statsmodels --upgrade --force-reinstall

# 等待statsmodels安装完成
time.sleep(10)

# 重新导入statsmodels
import statsmodels.api as sm

# 重新检查VIF
vif_data = pd.DataFrame()
vif_data['feature'] = features.columns
vif_data['VIF'] = [variance_inflation_factor(features.values, i) for i in range(len(features.columns))]

print(vif_data)

# 查看VIF值大于10的特征（通常认为VIF>10表示高度共线性）
high_vif_features = vif_data[vif_data['VIF'] > 10]['feature'].tolist()
print("Features with VIF > 10:", high_vif_features)





Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

bleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python3.10 from "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\Scripts\python.exe"
  * The NumPy version is: "1.24.3"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: DLL load failed while importing _multiarray_umath: 找不到指定的模块。
---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
Cell In[13], line 14
     12 time.sleep(10)
     13 # 重新导入statsmodels
---> 14 import statsmodels.api as sm
     15 # 重新检查VIF
     16 vif_data = pd.DataFrame()

File D:\LLM-Driven_AI

In [ ]:




# Uninstall NumPy
!pip uninstall numpy -y

# Reinstall NumPy
!pip install numpy

# Attempt to reinstall statsmodels
!pip install statsmodels --upgrade --force-reinstall

# Verify the installation of NumPy
import numpy
print(f'NumPy version: {numpy.__version__}')

# Try importing statsmodels
import statsmodels.api as sm




Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

ons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python3.10 from "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\Scripts\python.exe"
  * The NumPy version is: "1.24.3"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: DLL load failed while importing _multiarray_umath: 找不到指定的模块。
---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
Cell In[14], line 11
      9 print(f'NumPy version: {numpy.__version__}')
     10 # Try importing statsmodels
---> 11 import statsmodels.api as sm

File D:\LLM-Driven_AI

In [ ]:





# Uninstall NumPy
!{sys.executable} -m pip uninstall numpy -y

# Reinstall NumPy (version 1.21.5)
!{sys.executable} -m pip install numpy==1.21.5

# Reinstall statsmodels
!{sys.executable} -m pip install statsmodels --upgrade --force-reinstall

# Verify the installation of NumPy
import numpy
print(f'NumPy version: {numpy.__version__}')

# Try importing statsmodels
import statsmodels.api as sm
print(f'statsmodels version: {sm.__version__}')

# Import necessary functions
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Use the VIF to check for multicollinearity
vif_data = pd.DataFrame()
vif_data['feature'] = features.columns
vif_data['VIF'] = [variance_inflation_factor(features.values, i) for i in range(len(features.columns))]

# Display the VIF data
print(vif_data)

# Identify features with VIF > 10
high_vif_features = vif_data[vif_data['V IF'] > 10]['feature'].tolist()
print("Features with VIF > 10:", high_vif_features)





Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

ror.html

Please note and check the following:

  * The Python version is: Python3.10 from "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\Scripts\python.exe"
  * The NumPy version is: "1.24.3"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: DLL load failed while importing _multiarray_umath: 找不到指定的模块。
---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
Cell In[15], line 11
      9 print(f'NumPy version: {numpy.__version__}')
     10 # Try importing statsmodels
---> 11 import statsmodels.api as sm
     12 print(f'statsmodels version: {sm.__version__}')
     13 # Import necessary functions

File D:\LLM-Driven_AI

In [ ]:

# Uninstall NumPy
!{sys.executable} -m pip uninstall numpy -y
# Uninstall statsmodels
!{sys.executable} -m pip uninstall statsmodels -y


Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6
D:\software\Python\PyCharm\PyCharm 20250201\PyCharm 2025.2.0.1\plugins\python-ce\helpers\pycharm_display\datalore\display\supported_data_type.py:6: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  import numpy
You can safely remove it manually.
You can safely remove it manually.
Found existing installation: statsmodels 0.14.5
Uninstalling statsmodels-0.14.5:
  Successfully uninstalled statsmodels-0.14.5


In [ ]:


# Reinstall NumPy (version 1.24.3)
!{sys.executable} -m pip install numpy==1.24.3

# Reinstall statsmodels
!{sys.executable} -m pip install statsmodels --upgrade --force-reinstall

# Verify the installation
import numpy
print(f'NumPy version: {numpy.__version__}')
import statsmodels.api as sm
print(f'statsmodels version: {sm.__version__}')



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

tps://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python3.10 from "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\Scripts\python.exe"
  * The NumPy version is: "1.24.3"

and make sure that they are the versions you expect.
Please carefully study the documentation linked above for further help.

Original error was: DLL load failed while importing _multiarray_umath: 找不到指定的模块。
---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
Cell In[17], line 8
      6 import numpy
      7 print(f'NumPy version: {numpy.__version__}')
----> 8 import statsmodels.api as sm
      9 print(f'statsmodels version: {sm.__version__}')

File D:\LLM-Driven_AI

In [ ]:



# Uninstall NumPy using pip
!{sys.executable} -m pip uninstall numpy -y

# Reinstall NumPy using pip
!{sys.executable} -m pip install numpy

# Verify the installation
import numpy
print(f'NumPy version: {numpy.__version__}')




Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6
D:\software\Python\PyCharm\PyCharm 20250201\PyCharm 2025.2.0.1\plugins\python-ce\helpers\pycharm_display\datalore\display\supported_data_type.py:6: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  import numpy
You can safely remove it manually.
You can safely remove it manually.
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
NumPy version: 1.24.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-10, 10, 400)
y = f(x)

plt.plot(x, y, label=r'$f(x) = x^2 + 2x + 1$')
plt.scatter(0, 1, color='red', label='Critical Point (0, 1)')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Graph of $f(x)$')
plt.legend()
plt.grid(True)
plt.show()


---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
Cell In[19], line 4
      2 import numpy as np
      3 print('##active_line2##')
----> 4 import matplotlib.pyplot as plt
      5 print('##active_line3##')
      6 pass

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\matplotlib\__init__.py:159
    155 from packaging.version import parse as parse_version
    157 # cbook must import matplotlib only within function
    158 # definitions, so it is safe to import from it here.
--> 159 from . import _api, _version, cbook, _docstring, rcsetup
    160 from matplotlib.cbook import sanitize_sequence
    161 from matplotlib._api import MatplotlibDeprecationWarning

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\matplotlib\rcsetup.py:26
     23 import numpy as np
     25 from matplotlib import _api, cbook
---> 26 from 

In [ ]:



# Uninstall and reinstall matplotlib
!{sys.executable} -m pip uninstall matplotlib -y
!{sys.executable} -m pip install matplotlib --prefer-binary

# Verify the installation
import matplotlib
print(f'matplotlib version: {matplotlib.__version__}')



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

t\experiments\openinterpreter\venv\lib\site-packages (from matplotlib) (3.2.4)
D:\software\Python\PyCharm\PyCharm 20250201\PyCharm 2025.2.0.1\plugins\python-ce\helpers\pycharm_display\datalore\display\supported_data_type.py:6: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  import numpy
---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
Cell In[20], line 5
      3 get_ipython().system('{sys.executable} -m pip install matplotlib --prefer-binary')
      4 # Verify the installation
----> 5 import matplotlib
      6 print(f'matplotlib version: {matplotlib.__version__}')

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterp

In [ ]:


import matplotlib.pyplot as plt



---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
Cell In[21], line 2
      1 print('##active_line1##')
----> 2 import matplotlib.pyplot as plt

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\matplotlib\__init__.py:159
    155 from packaging.version import parse as parse_version
    157 # cbook must import matplotlib only within function
    158 # definitions, so it is safe to import from it here.
--> 159 from . import _api, _version, cbook, _docstring, rcsetup
    160 from matplotlib.cbook import sanitize_sequence
    161 from matplotlib._api import MatplotlibDeprecationWarning

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\matplotlib\rcsetup.py:26
     23 import numpy as np
     25 from matplotlib import _api, cbook
---> 26 from matplotlib.backends import BackendFilter, backend_registry
     27 from ma